**Sample ID**: 55

**Query**:

I want to change all my pending t-shirt orders to a purple, size S, v-neck, polyester shirt.

**DB Type**: Base Case

**Case Description**:

The user, Yusuf Rossi (zip code 19122), is identified by his name and zip code. He wants to modify items in his pending orders. The system identifies two pending orders, #W6247578 and #W4776164, that contain t-shirts. The user wants to replace the existing t-shirts in these orders (item IDs 3799046073 and 8349118980) with the t-shirt variant (item ID 9647292434, under product 9523456873) that is purple, size S, v-neck, and made of polyester. Any price difference for the modification is to be charged to his credit card credit_card_9513926.





```
<multiturn info>
User Identification: Yusuf Rossi, Zip 19122 (Information Gathering)
Initial Inquiry: How many t-shirt options are available (Information Gathering)
Task: Modify all pending t-shirt orders (Information Gathering)
New T-shirt Color: Purple (Information Gathering)
New T-shirt Size: S (Information Gathering)
New T-shirt Style: V-neck (Information Gathering)
New T-shirt Material: Polyester (Information Gathering)
</multiturn info>
```

**Global/Context Variables:**


**APIs:**

- retail


# Set Up

## Download relevant files

In [23]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [24]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [25]:
# Using default DB for the Tau Benchmark
import retail

retail.SimulationEngine.db.load_state("/content/DBs/RetailDefaultDB.json")


# Initial Assertion

1. A user exists with the first name `Yusuf`, last name `Rossi`, and zip code `19122`, corresponding to user ID `yusuf_rossi_9620`.
2. The user `yusuf_rossi_9620` has at least two orders, `#W6247578` and `#W4776164`.
3. Order `#W6247578` has a `pending` status and contains a t-shirt with item ID `3799046073`.
4. Order `#W4776164` has a `pending` status and contains a t-shirt with item ID `8349118980`.
5. A product with ID `9523456873` has an item with ID `9647292434`, which is available and consists of the options: color `purple`, size `S`, style `v-neck`, and material `polyester`.
6. The user `yusuf_rossi_9620` has an associated payment method with ID `credit_card_9513926`.

In [26]:
import retail
import json
from retail.SimulationEngine.custom_errors import ProductNotFoundError
from Scripts.assertions_utils import *

# --- Constants for Initial Assertions ---
USER_FIRST_NAME = 'Yusuf'
USER_LAST_NAME = 'Rossi'
USER_ZIP_CODE = '19122'
USER_ID = 'yusuf_rossi_9620'
ORDER_ID_1 = '#W6247578'
ORDER_ID_2 = '#W4776164'
ITEM_ID_1 = '3799046073'
ITEM_ID_2 = '8349118980'
PRODUCT_ID = '9523456873'
NEW_ITEM_ID = '9647292434'
PAYMENT_METHOD_ID = 'credit_card_9513926'
EXPECTED_PRODUCT_OPTIONS = {
    'color': 'purple',
    'size': 'S',
    'style': 'v-neck',
    'material': 'polyester'
}
PENDING_STATUS = 'pending'

# --- Assertion 1: A user exists with the first name 'Yusuf', last name 'Rossi', and zip code '19122', corresponding to user ID 'yusuf_rossi_9620'. ---
found_user_id = None
api_call_error_1 = None
try:
    found_user_id = retail.find_user_id_by_name_zip(
        first_name=USER_FIRST_NAME,
        last_name=USER_LAST_NAME,
        zip_code=USER_ZIP_CODE
    )
except Exception as e:
    api_call_error_1 = str(e)

assertion_message_1 = f"Assertion 1 Failed: Expected to find user ID '{USER_ID}' for user '{USER_FIRST_NAME} {USER_LAST_NAME}' with zip '{USER_ZIP_CODE}'. "
if api_call_error_1:
    assertion_message_1 += f"API call failed: {api_call_error_1}."
else:
    assertion_message_1 += f"Instead, found user ID: '{found_user_id}'."
assertion_condition_1 = compare_strings(found_user_id, USER_ID)
assert assertion_condition_1, assertion_message_1

# --- Pre-computation for Assertions 2 & 6: Get user details ---
user_details = None
api_call_error_2_6 = None
try:
    user_details = retail.get_user_details(user_id=USER_ID)
except Exception as e:
    api_call_error_2_6 = str(e)

# --- Assertion 2: The user 'yusuf_rossi_9620' has at least two orders, '#W6247578' and '#W4776164'. ---
user_orders = []
if user_details:
    user_orders = user_details.get('orders', [])

assertion_message_2 = f"Assertion 2 Failed: User '{USER_ID}' should have orders '{ORDER_ID_1}' and '{ORDER_ID_2}'. "
if api_call_error_2_6:
    assertion_message_2 += f"API call to get user details failed: {api_call_error_2_6}."
else:
    assertion_message_2 += f"Actual orders found: {user_orders}."
assertion_condition_2 = (compare_is_list_subset(ORDER_ID_1, user_orders) and compare_is_list_subset(ORDER_ID_2, user_orders))
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Order '#W6247578' has a 'pending' status and contains a t-shirt with item ID '3799046073'. ---
order_1_details = None
api_call_error_3 = None
try:
    order_1_details = retail.get_order_details(order_id=ORDER_ID_1)
except Exception as e:
    api_call_error_3 = str(e)

order_1_status_is_pending = False
order_1_item_found = False
if order_1_details:
    order_1_status_is_pending = compare_strings(order_1_details.get('status'), PENDING_STATUS)
    order_1_items = order_1_details.get('items', [])
    order_1_item_found = any(compare_strings(item.get('item_id'), ITEM_ID_1) for item in order_1_items)

assertion_message_3 = f"Assertion 3 Failed: Order '{ORDER_ID_1}' should be 'pending' and contain item '{ITEM_ID_1}'. "
if api_call_error_3:
    assertion_message_3 += f"API call failed: {api_call_error_3}."
else:
    assertion_message_3 += f"Actual status: '{order_1_details.get('status', 'N/A')}', Item found: {order_1_item_found}."
assertion_condition_3 = order_1_status_is_pending and order_1_item_found
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: Order '#W4776164' has a 'pending' status and contains a t-shirt with item ID '8349118980'. ---
order_2_details = None
api_call_error_4 = None
try:
    order_2_details = retail.get_order_details(order_id=ORDER_ID_2)
except Exception as e:
    api_call_error_4 = str(e)

order_2_status_is_pending = False
order_2_item_found = False
if order_2_details:
    order_2_status_is_pending = compare_strings(order_2_details.get('status'), PENDING_STATUS)
    order_2_items = order_2_details.get('items', [])
    order_2_item_found = any(compare_strings(item.get('item_id'), ITEM_ID_2) for item in order_2_items)

assertion_message_4 = f"Assertion 4 Failed: Order '{ORDER_ID_2}' should be 'pending' and contain item '{ITEM_ID_2}'. "
if api_call_error_4:
    assertion_message_4 += f"API call failed: {api_call_error_4}."
else:
    assertion_message_4 += f"Actual status: '{order_2_details.get('status', 'N/A')}', Item found: {order_2_item_found}."
assertion_condition_4 = order_2_status_is_pending and order_2_item_found
assert assertion_condition_4, assertion_message_4

# --- Assertion 5: Product '9523456873' has an available item '9647292434' with options color 'purple', size 'S', style 'v-neck', and material 'polyester'. ---
product_details = None
variant_found = False
try:
    product_details = retail.get_product_details(product_id=PRODUCT_ID)
    variants = product_details.get('variants', {})
    for item_id, variant_info in variants.items():
        if (compare_strings(item_id, NEW_ITEM_ID)
            and variant_info.get('available')
            and variant_info.get('options') == EXPECTED_PRODUCT_OPTIONS):
            variant_found = True
            break
except ProductNotFoundError:
    pass

assert variant_found, (
    f"Assertion 5 Failed: Product '{PRODUCT_ID}' should have an available "
    f"variant with item id '{NEW_ITEM_ID}' and options {EXPECTED_PRODUCT_OPTIONS}."
)

# --- Assertion 6: The user 'yusuf_rossi_9620' has an associated payment method with ID 'credit_card_9513926'. ---
payment_method_found = False
if user_details:
    payment_methods = user_details.get('payment_methods', {})
    payment_method_found = compare_is_list_subset(PAYMENT_METHOD_ID, list(payment_methods.keys()))

assert payment_method_found, (
    f"Assertion 6 Failed: User '{USER_ID}' should have payment method '{PAYMENT_METHOD_ID}'. "
    f"Actual methods: {list(user_details.get('payment_methods', {}).keys()) if user_details else []}."
)


# Action

**Simulated User**: I want to know how many t-shirt options are available.

**Action Agent**: Sure, I can help you with that. For security, could you confirm your name and zip code?

**Simulated User**: Okay, it's Yusuf Rossi, zip 19122 .

In [27]:
import retail
retail.find_user_id_by_name_zip(first_name='Yusuf', last_name='Rossi', zip_code='19122')

'yusuf_rossi_9620'

In [28]:
retail.list_all_product_types()

{'products': {'Action Camera': '3377618313',
  'Air Purifier': '3821016478',
  'Backpack': '2524789262',
  'Bicycle': '9783735446',
  'Bluetooth Speaker': '4768869376',
  'Bookshelf': '8600330539',
  'Coffee Maker': '7996920482',
  'Cycling Helmet': '7765186836',
  'Desk Lamp': '6817146515',
  'Digital Camera': '8940227892',
  'Dumbbell Set': '7233192239',
  'E-Reader': '3801771308',
  'Electric Kettle': '1075968781',
  'Electric Toothbrush': '7352963235',
  'Espresso Machine': '4354588079',
  'Fleece Jacket': '8560156827',
  'Gaming Mouse': '5713490933',
  'Garden Hose': '6679515468',
  'Grill': '6819683148',
  'Headphones': '6992792935',
  'Hiking Boots': '7363354090',
  'Indoor Security Camera': '2985987096',
  'Jigsaw Puzzle': '1808611083',
  'LED Light Bulb': '2696197613',
  'Laptop': '4760268021',
  'Luggage Set': '5426915165',
  'Makeup Kit': '5149340237',
  'Mechanical Keyboard': '1656367028',
  'Notebook': '2892623495',
  'Office Chair': '4794339885',
  'Patio Umbrella': '9743

In [29]:
retail.get_product_details(product_id='9523456873')

{'name': 'T-Shirt',
 'product_id': '9523456873',
 'variants': {'9612497925': {'item_id': '9612497925',
   'options': {'color': 'blue',
    'size': 'M',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 50.88},
  '8124970213': {'item_id': '8124970213',
   'options': {'color': 'purple',
    'size': 'XL',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 49.67},
  '9354168549': {'item_id': '9354168549',
   'options': {'color': 'red',
    'size': 'XXL',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 46.85},
  '5253880258': {'item_id': '5253880258',
   'options': {'color': 'black',
    'size': 'XXL',
    'material': 'polyester',
    'style': 'v-neck'},
   'available': True,
   'price': 49.52},
  '1176194968': {'item_id': '1176194968',
   'options': {'color': 'black',
    'size': 'S',
    'material': 'polyester',
    'style': 'crew neck'},
   'available': True,
   'price': 52.88}

**Action Agent**: There are 10 different T-shirt options available.
Here are the available options:

- blue / M / cotton / crew neck — `$50.88`
- purple / XL / cotton / crew neck — `$49.67`
- red / XXL / cotton / crew neck — `$46.85`
- black / XXL / polyester / v-neck — `$49.52`
- black / S / polyester / crew neck — `$52.88`
- purple / S / polyester / v-neck — `$53.48`
- blue / S / cotton / v-neck — `$53.43`
- black / XXL / cotton / crew neck — `$53.27`
- red / L / cotton / v-neck — `$46.66`
- black / XL / cotton / crew neck — `$51.05`


**Simulated User**: Okay, I want to change all my pending t-shirt orders to a purple, size S, v-neck, polyester shirt.

**Action Agent**: Sure, I can help you with that.

In [30]:
retail.get_user_details(user_id='yusuf_rossi_9620')

{'name': {'first_name': 'Yusuf', 'last_name': 'Rossi'},
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'email': 'yusuf.rossi7301@example.com',
 'payment_methods': {'credit_card_9513926': {'id': 'credit_card_9513926',
   'source': 'credit_card',
   'brand': 'mastercard',
   'last_four': '2478'}},
 'orders': ['#W6247578', '#W9711842', '#W4776164', '#W6679257', '#W2378156']}

In [31]:
retail.get_order_details(order_id='#W6247578')

{'order_id': '#W6247578',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'T-Shirt',
   'product_id': '9523456873',
   'item_id': '3799046073',
   'price': 53.27,
   'options': {'color': 'black',
    'size': 'XXL',
    'material': 'cotton',
    'style': 'crew neck'}}],
 'fulfillments': [],
 'status': 'pending',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 53.27,
   'payment_method_id': 'credit_card_9513926'}]}

In [32]:
retail.get_product_details(product_id='9523456873')

{'name': 'T-Shirt',
 'product_id': '9523456873',
 'variants': {'9612497925': {'item_id': '9612497925',
   'options': {'color': 'blue',
    'size': 'M',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 50.88},
  '8124970213': {'item_id': '8124970213',
   'options': {'color': 'purple',
    'size': 'XL',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 49.67},
  '9354168549': {'item_id': '9354168549',
   'options': {'color': 'red',
    'size': 'XXL',
    'material': 'cotton',
    'style': 'crew neck'},
   'available': True,
   'price': 46.85},
  '5253880258': {'item_id': '5253880258',
   'options': {'color': 'black',
    'size': 'XXL',
    'material': 'polyester',
    'style': 'v-neck'},
   'available': True,
   'price': 49.52},
  '1176194968': {'item_id': '1176194968',
   'options': {'color': 'black',
    'size': 'S',
    'material': 'polyester',
    'style': 'crew neck'},
   'available': True,
   'price': 52.88}

In [33]:
retail.get_order_details(order_id='#W9711842')

{'order_id': '#W9711842',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'Sunglasses',
   'product_id': '7314138884',
   'item_id': '4245201809',
   'price': 281.48,
   'options': {'frame color': 'black',
    'lens color': 'green',
    'lens type': 'non-polarized',
    'frame material': 'metal'}}],
 'fulfillments': [{'tracking_id': ['540934230326'],
   'item_ids': ['4245201809']}],
 'status': 'cancelled',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 281.48,
   'payment_method_id': 'credit_card_9513926'},
  {'transaction_type': 'refund',
   'amount': 281.48,
   'payment_method_id': 'credit_card_9513926'}]}

In [34]:
retail.get_order_details(order_id='#W4776164')

{'order_id': '#W4776164',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'T-Shirt',
   'product_id': '9523456873',
   'item_id': '8349118980',
   'price': 53.43,
   'options': {'color': 'blue',
    'size': 'S',
    'material': 'cotton',
    'style': 'v-neck'}},
  {'name': 'Espresso Machine',
   'product_id': '4354588079',
   'item_id': '6324294385',
   'price': 2719.01,
   'options': {'pressure': '9 bar', 'capacity': '1L', 'type': 'automatic'}}],
 'fulfillments': [],
 'status': 'pending',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 2772.44,
   'payment_method_id': 'credit_card_9513926'}]}

In [35]:
retail.get_order_details(order_id='#W6679257')

{'order_id': '#W6679257',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'Digital Camera',
   'product_id': '8940227892',
   'item_id': '5996159312',
   'price': 2895.55,
   'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}}],
 'fulfillments': [{'tracking_id': ['522129247270'],
   'item_ids': ['5996159312']}],
 'status': 'delivered',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 2895.55,
   'payment_method_id': 'credit_card_9513926'}]}

In [36]:
retail.get_order_details(order_id='#W2378156')

{'order_id': '#W2378156',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'Headphones',
   'product_id': '6992792935',
   'item_id': '4202497723',
   'price': 342.81,
   'options': {'type': 'over-ear',
    'connectivity': 'wireless',
    'color': 'blue'}},
  {'name': 'Vacuum Cleaner',
   'product_id': '1762337868',
   'item_id': '4602305039',
   'price': 561.05,
   'options': {'type': 'robotic',
    'bagged/bagless': 'bagged',
    'features': 'cordless'}},
  {'name': 'Mechanical Keyboard',
   'product_id': '1656367028',
   'item_id': '1151293680',
   'price': 272.33,
   'options': {'switch type': 'linear',
    'backlight': 'RGB',
    'size': 'full size'}},
  {'name': 'Smart Thermostat',
   'product_id': '4896585277',
   'item_id': '4983901480',
   'price': 262.47,
   'options': {'compatibility': 'Apple HomeKit', 'color': 'black'}},
  

In [37]:
retail.calculate(expression='53.48 - 53.27')

'0.21'

In [38]:
retail.calculate(expression='53.48 - 53.43')

'0.05'

**Action Agent**: I have identified two pending orders with T-shirts, #W6247578 and #W4776164.

For order #W6247578, I will exchange the current T-shirt (item ID 3799046073, black, XXL, cotton, crew neck, price \$53.27) for a new T-shirt (item ID 9647292434, purple, S, polyester, v-neck, price \$53.48). The price difference is \$0.21.

For order #W4776164, I will exchange the current T-shirt (item ID 8349118980, blue, S, cotton, v-neck, price \$53.43) for the same new T-shirt (item ID 9647292434, purple, S, polyester, v-neck, price \$53.48). The price difference is \$0.05.

The total cost for these changes will be \$0.26, which will be charged to your credit card ending in 2478.

Do you want to proceed with these changes?

**Simulated User**: Yes, please proceed with these changes.

In [39]:
retail.modify_pending_order_items(order_id='#W6247578', item_ids=['3799046073'], new_item_ids=['9647292434'], payment_method_id='credit_card_9513926')

{'order_id': '#W6247578',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'T-Shirt',
   'product_id': '9523456873',
   'item_id': '9647292434',
   'price': 53.48,
   'options': {'color': 'purple',
    'size': 'S',
    'material': 'polyester',
    'style': 'v-neck'}}],
 'fulfillments': [],
 'status': 'pending (item modified)',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 53.27,
   'payment_method_id': 'credit_card_9513926'},
  {'transaction_type': 'payment',
   'amount': 0.20999999999999375,
   'payment_method_id': 'credit_card_9513926'}]}

In [40]:
retail.modify_pending_order_items(order_id='#W4776164', item_ids=['8349118980'], new_item_ids=['9647292434'], payment_method_id='credit_card_9513926')

{'order_id': '#W4776164',
 'user_id': 'yusuf_rossi_9620',
 'address': {'address1': '763 Broadway',
  'address2': 'Suite 135',
  'city': 'Philadelphia',
  'country': 'USA',
  'state': 'PA',
  'zip': '19122'},
 'items': [{'name': 'T-Shirt',
   'product_id': '9523456873',
   'item_id': '9647292434',
   'price': 53.48,
   'options': {'color': 'purple',
    'size': 'S',
    'material': 'polyester',
    'style': 'v-neck'}},
  {'name': 'Espresso Machine',
   'product_id': '4354588079',
   'item_id': '6324294385',
   'price': 2719.01,
   'options': {'pressure': '9 bar', 'capacity': '1L', 'type': 'automatic'}}],
 'fulfillments': [],
 'status': 'pending (item modified)',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 2772.44,
   'payment_method_id': 'credit_card_9513926'},
  {'transaction_type': 'payment',
   'amount': 0.04999999999999716,
   'payment_method_id': 'credit_card_9513926'}]}

**Action Agent**: I have successfully updated both of your pending T-shirt orders, #W6247578 and #W4776164, to the new purple, size S, v-neck, polyester shirt. Your credit card has been charged a total of \$0.26 for the price difference.

# Final Assertion

1. The order `#W6247578` contains an item with ID `9647292434` and options: color `purple`, size `S`, style `v-neck`, and material `polyester`.
2. The order `#W4776164` contains an item with ID `9647292434` and options: color `purple`, size `S`, style `v-neck`, and material `polyester`.
3. The payment history for order `#W6247578` shows a new transaction for the price difference charged to payment method `credit_card_9513926`.
4. The payment history for order `#W4776164` shows a new transaction for the price difference charged to payment method `credit_card_9513926`.

In [41]:
import retail
import json
from Scripts.assertions_utils import *

# --- Constants for the test case ---
ORDER_ID_1 = "#W6247578"
ORDER_ID_2 = "#W4776164"
NEW_ITEM_ID = "9647292434"
PAYMENT_METHOD_ID = "credit_card_9513926"
EXPECTED_OPTIONS = {
    "color": "purple",
    "size": "S",
    "style": "v-neck",
    "material": "polyester"
}

# --- Fixed Helper function to find the new item ---
def find_new_item(items, expected_item_id, expected_options):
    for item in items:
        # Correctly check the item_id and options
        if compare_strings(item.get('item_id'), expected_item_id):
            if item.get('options') == expected_options:
                return item
    return None

# --- Helper function to check for transaction ---
def find_transaction(payment_history, payment_method_id):
    for transaction in payment_history:
        if compare_strings(transaction.get('payment_method_id'), payment_method_id):
            if transaction.get('amount', 0) > 0:
                return transaction
    return None

# --- Data Gathering ---
order_1_details = None
order_2_details = None
api_error = None

try:
    order_1_details = retail.get_order_details(order_id=ORDER_ID_1)
    order_2_details = retail.get_order_details(order_id=ORDER_ID_2)
except Exception as e:
    api_error = str(e)

# --- Assertion 1 ---
new_item_in_order_1 = None
if order_1_details and 'items' in order_1_details:
    items_1 = order_1_details.get('items', [])
    new_item_in_order_1 = find_new_item(items_1, NEW_ITEM_ID, EXPECTED_OPTIONS)

assertion_message_1 = (
    f"API Error: {api_error}. "
    f"New item found: {'Yes' if new_item_in_order_1 else 'No'}. "
    f"Expected options: {EXPECTED_OPTIONS}. "
    f"Actual options: {new_item_in_order_1.get('options') if new_item_in_order_1 else 'N/A'}."
)
assertion_condition_1 = new_item_in_order_1 is not None
assert assertion_condition_1, assertion_message_1

# --- Assertion 2 ---
new_item_in_order_2 = None
if order_2_details and 'items' in order_2_details:
    items_2 = order_2_details.get('items', [])
    new_item_in_order_2 = find_new_item(items_2, NEW_ITEM_ID, EXPECTED_OPTIONS)

assertion_message_2 = (
    f"API Error: {api_error}. "
    f"New item found: {'Yes' if new_item_in_order_2 else 'No'}. "
    f"Expected options: {EXPECTED_OPTIONS}. "
    f"Actual options: {new_item_in_order_2.get('options') if new_item_in_order_2 else 'N/A'}."
)
assertion_condition_2 = new_item_in_order_2 is not None
assert assertion_condition_2, assertion_message_2

# --- Assertion 3 ---
transaction_in_order_1 = None
if order_1_details and 'payment_history' in order_1_details:
    payment_history_1 = order_1_details.get('payment_history', [])
    transaction_in_order_1 = find_transaction(payment_history_1, PAYMENT_METHOD_ID)

assertion_message_3 = (
    f"Assertion 3 Failed: No new charge transaction found for order '{ORDER_ID_1}' with payment method '{PAYMENT_METHOD_ID}'. "
    f"API Error: {api_error}. "
    f"Payment history: {json.dumps(order_1_details.get('payment_history', [])) if order_1_details else 'N/A'}."
)
assertion_condition_3 = transaction_in_order_1 is not None
assert assertion_condition_3, assertion_message_3

# --- Assertion 4 ---
transaction_in_order_2 = None
if order_2_details and 'payment_history' in order_2_details:
    payment_history_2 = order_2_details.get('payment_history', [])
    transaction_in_order_2 = find_transaction(payment_history_2, PAYMENT_METHOD_ID)

assertion_message_4 = (
    f"Assertion 4 Failed: No new charge transaction found for order '{ORDER_ID_2}' with payment method '{PAYMENT_METHOD_ID}'. "
    f"API Error: {api_error}. "
    f"Payment history: {json.dumps(order_2_details.get('payment_history', [])) if order_2_details else 'N/A'}."
)
assertion_condition_4 = transaction_in_order_2 is not None
assert assertion_condition_4, assertion_message_4
